# Oxford-IIIT Pet subset: frozen-split EDA

Run this only after `scripts/prepare_oxford_pets.py`. This notebook reads frozen manifests and reports; it does not create or modify a split. No model output is used, and augmentation examples come from **train only**.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'splits').exists())
sys.path.insert(0, str(ROOT))
DATASET_ROOT = ROOT / 'data' / 'oxford-iiit-pet'
SPLIT_DIR = ROOT / 'splits'
assert DATASET_ROOT.is_dir(), f'Missing local data: {DATASET_ROOT}'
assert (SPLIT_DIR / 'train.csv').is_file(), 'Run preparation first.'


In [ ]:
mapping = json.loads((SPLIT_DIR / 'class_mapping.json').read_text(encoding='utf-8'))
display(pd.DataFrame(mapping['classes']))
splits = {name: pd.read_csv(SPLIT_DIR / f'{name}.csv') for name in ('train', 'val', 'test')}
all_rows = pd.concat([frame.assign(split=name) for name, frame in splits.items()], ignore_index=True)
all_rows.head()

## Class sizes before and after exact-duplicate cleanup

In [ ]:
integrity = json.loads((SPLIT_DIR / 'integrity_report.json').read_text(encoding='utf-8'))
counts = pd.DataFrame({'before_exact_dedup': integrity['counts_before_exact_dedup'],
                       'after_exact_dedup': integrity['counts_after_exact_dedup']}).fillna(0).astype(int)
display(counts)
counts.plot.bar(figsize=(12, 4), ylabel='images', title='Selected annotated images before/after exact de-duplication');
plt.xticks(rotation=35, ha='right'); plt.tight_layout()

## Frozen split distribution and leakage checks

In [ ]:
distribution = pd.crosstab(all_rows['class_name'], all_rows['split']).reindex(columns=['train', 'val', 'test'], fill_value=0)
display(distribution)
distribution.plot.bar(stacked=True, figsize=(12, 4), ylabel='images', title='Class distribution in frozen custom split');
plt.xticks(rotation=35, ha='right'); plt.tight_layout()

for field in ('image_id', 'relative_path', 'content_hash'):
    sets = {name: set(frame[field]) for name, frame in splits.items()}
    assert not (sets['train'] & sets['val'] or sets['train'] & sets['test'] or sets['val'] & sets['test']), field
print('OK: train/val/test do not intersect by image_id, relative_path, or content_hash.')
display(pd.Series({name: len(frame) for name, frame in splits.items()}, name='images'))

## Integrity and duplicate reports

Near-duplicate entries are candidates for manual review only; they are not removed based on perceptual-hash distance.

In [ ]:
exact = json.loads((SPLIT_DIR / 'exact_duplicates.json').read_text(encoding='utf-8'))
near = json.loads((SPLIT_DIR / 'near_duplicate_candidates.json').read_text(encoding='utf-8'))
print({'annotation_candidates': integrity['annotation_candidates'], 'valid_decoded': integrity['valid_decoded'],
       'retained_after_exact_dedup': integrity['retained_after_exact_dedup'],
       'excluded': len(integrity['excluded']), 'exact_duplicate_groups': len(exact['groups']),
       'near_duplicate_candidates': len(near['candidates'])})
display(pd.DataFrame(integrity['excluded']))
display(pd.DataFrame(exact['groups']))
display(pd.DataFrame(near['candidates']).head(50))

## Human review status for perceptual-hash candidates

This is a data-integrity workflow only. The decisions below must not be used for preprocessing, split, class, hyperparameter, or model-condition decisions. `uncertain` remains open for local visual inspection.

In [ ]:
review_path = SPLIT_DIR / 'near_duplicate_review.csv'
status_path = SPLIT_DIR / 'near_duplicate_review_status.json'
assert review_path.is_file() and status_path.is_file(), 'Build the queue: py -3 scripts/review_near_duplicates.py'
review = pd.read_csv(review_path)
review_status = json.loads(status_path.read_text(encoding='utf-8'))
display(review['decision'].value_counts(dropna=False).rename_axis('decision').to_frame('pairs'))
display(review[review['decision'].isin(['duplicate', 'near-duplicate'])])
crossings = pd.DataFrame(review_status['confirmed_cross_split_pairs'])
display(crossings if not crossings.empty else pd.DataFrame({'status': ['No confirmed cross-split pairs.']}))
unresolved_crossings = pd.DataFrame(review_status['unresolved_cross_split_candidates'])
print(f'Unresolved cross-split candidates requiring visual review: {len(unresolved_crossings)}')
print('For local side-by-side inspection, open:', SPLIT_DIR / 'near_duplicate_review.html')

## Image dimensions and aspect ratios

In [ ]:
def dimensions(relative_path):
    with Image.open(DATASET_ROOT / relative_path) as image:
        return image.size

sizes = all_rows['relative_path'].map(dimensions)
all_rows[['width', 'height']] = pd.DataFrame(sizes.tolist(), index=all_rows.index)
all_rows['aspect_ratio'] = all_rows['width'] / all_rows['height']
display(all_rows[['width', 'height', 'aspect_ratio']].describe())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(all_rows['width'], bins=30, ax=axes[0], label='width', color='steelblue')
sns.histplot(all_rows['height'], bins=30, ax=axes[0], label='height', color='orange'); axes[0].legend()
sns.histplot(all_rows['aspect_ratio'], bins=30, ax=axes[1]); axes[1].set_title('Aspect ratio (width/height)')
plt.tight_layout()

## Train-only augmentation examples

These samples illustrate the fixed training augmentation. No validation or test image is displayed here.

In [ ]:
import torch
from src.dataset import PetManifestDataset, _train_transform

train_dataset = PetManifestDataset(DATASET_ROOT, SPLIT_DIR / 'train.csv', SPLIT_DIR / 'class_mapping.json', _train_transform())
mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, index in zip(axes.flat, range(8)):
    tensor, label = train_dataset[index]
    image = (tensor * std + mean).clamp(0, 1).permute(1, 2, 0)
    axis.imshow(image)
    axis.set_title(train_dataset.mapping[label])
    axis.axis('off')
plt.tight_layout()